# sre-gym Advanced tier — blueprint walkthrough

> The Advanced tier is **not runnable** in this repo. This notebook walks through the design at the YAML level so you can see what a horizon-bounded scenario looks like end-to-end.

We load each of the three reference scenarios, render their topology / incident chain / reward dimensions / reference trace, and explain what a downstream operator would need to build to make it runnable.

This notebook is the artifact that defends the claim *"Advanced is documented as a roadmap, not a marketing line."* Every YAML field is real and structured; if you wanted to lift this tier into a 1-2 A100-day training run, the spec is here.

In [ ]:
%%bash
pip install -q pyyaml openenv-core>=0.2.1 pydantic>=2.8 fastapi httpx
if [ ! -d /content/sre-enginnerllm ]; then
  git clone https://github.com/dakshdoesdev/sre-enginnerllm.git /content/sre-enginnerllm
fi

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, '/content/sre-enginnerllm')

from sre_gym import SREGym, Tier

env = SREGym(tier=Tier.ADVANCED)
print('Tier description:')
for k, v in env.describe().items():
    print(f'  {k:<30} {v}')

## 1. Scenario 1 — `cascading_release_train`

Multi-stage incident: a release train deploys 3 services together. Phase-1 fix triggers a phase-2 incident 25 ticks later. Tests long-horizon state tracking and recovery from early mistakes.

In [ ]:
import yaml
specs = env.list_scenarios()
by_id = {s['id']: s for s in specs}
print(f'Available reference scenarios: {list(by_id.keys())}')
spec = by_id['cascading_release_train']
print(f"\nName : {spec['name']}")
print(f'Difficulty: {spec["difficulty"]}')
print(f'\nDescription:\n{spec["description"]}')

In [ ]:
# Topology: 18 services, owners, SLOs.
import pandas as pd
topology_df = pd.DataFrame(spec['topology']['services'])
print(f'Topology: {len(topology_df)} services')
print(topology_df.to_string(index=False))

In [ ]:
# Incident chain — the load-bearing piece for the tier.
for phase in spec['incident_chain']:
    print(f"--- Phase {phase['phase']} ---")
    print(f"  starts_at_tick   : {phase['starts_at_tick']}")
    print(f"  triggered_by     : {phase['triggered_by']}")
    print(f"  failing_services : {phase['failing_services']}")
    print(f"  correct_action   : {phase['correct_action']}")
    print(f"  deceptive_signal : {phase['deceptive_signal']}")
    print()

In [ ]:
# 28 actions vs 11 in Basic; the new ones are listed explicitly.
advanced_only = set(spec['allowed_actions']) - {
    'query_logs','query_metrics','query_dependencies','query_deploys',
    'rollback_deploy','restart_service','run_check','isolate_service',
    'escalate','submit_hypothesis','declare_resolved'
}
print(f'Advanced-only actions ({len(advanced_only)}):')
for a in sorted(advanced_only):
    print(f'  {a}')

In [ ]:
# Reward dimensions — basic 7 + Advanced extensions.
for dim, meta in spec['reward_dimensions'].items():
    print(f'  {dim:<35} weight={meta["weight"]:+.2f} range={meta["range"]}')

In [ ]:
# Reference trace — the canonical optimal path.
for phase_name, steps in spec['reference_trace'].items():
    print(f'--- {phase_name} ---')
    for step in steps:
        print(f'  tick {step["tick"]:>3}: {step["action"]:<70} -> {step["expected_signal"]}')
    print()

## 2. Scenario 2 — `observability_pipeline_outage`

Partial observability — `query_logs` returns degraded data because the logging pipeline is the affected service. Tests alternate-observability-path use.

In [ ]:
spec = by_id['observability_pipeline_outage']
print(f"Name : {spec['name']}")
print(f'\nDescription:\n{spec["description"]}\n')
for phase in spec['incident_chain']:
    print(f"Phase {phase['phase']}: {phase['triggered_by']} -> {phase['correct_action']}")
    print(f"  decoy: {phase['deceptive_signal'][:120]}...")
    print()

## 3. Scenario 3 — `supabase_rls_silent_leak`

Cross-domain reasoning. A reliability incident with a security root cause. Tests containment-first discipline + leak-window quantification + customer-comm drafting. **No existing SRE benchmark scores this combination.**

In [ ]:
spec = by_id['supabase_rls_silent_leak']
print(f"Name : {spec['name']}")
print(f'\nDescription:\n{spec["description"]}\n')
print('Reward dimensions emphasizing the cross-domain lesson:')
for dim, meta in spec['reward_dimensions'].items():
    if meta['weight'] >= 0.10:
        print(f'  {dim:<35} weight={meta["weight"]:+.2f}')
print()
print('Success criteria:')
for crit in spec['success_criteria']:
    for k, v in crit.items():
        print(f'  - {k}: {v}')

## 4. What it would take to run these end-to-end

A faithful Advanced simulator backing these YAMLs needs:

1. A 15-20 service event-loop simulator (vs. the 4-service Basic one).
2. Multi-tick fault propagation with configurable causal latency between phases.
3. A synthetic on-call peer model with calibrated `correct_pct` per behaviour entry.
4. ~28 action handlers (the additions on top of Basic's 11).
5. A learned-critic reward path for `postmortem_quality` evaluation.
6. Time-pressure SLO countdowns surfacing in the observation.

Estimated effort: 2 weeks of focused engineering + 1-2 A100-days of training.

Trying to run any of these scenarios via the SDK gracefully degrades to a `TierNotRunnableError`:

In [ ]:
from sre_gym.env import TierNotRunnableError
try:
    env.reset()
except TierNotRunnableError as e:
    print(f'TierNotRunnableError caught: {e}')
    print(f'docs_path: {e.docs_path}')

## 5. Where this fits in the tier story

Basic teaches single-incident reasoning under tight compute. Advanced teaches multi-incident horizon coherence. Max teaches realism-bounded operations against live infrastructure. The dimensional escalation (compute → horizon → realism) is *the* claim.

If you're a startup CTO with 1-2 A100-days: the YAMLs in this notebook are a literal blueprint for what to train against. Start with `cascading_release_train`, then tackle `observability_pipeline_outage`, then graduate to `supabase_rls_silent_leak`.